In [1]:
from ingestion_pipeline_2 import IngPipeline

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import os
import sys

In [3]:
ing_configs = {
    'chunk_size': 400,
    'embed_model': "intfloat/e5-large-v2",
    'chunking_approach': "recursive",
    'kbs_path': "./test_data/kbs",
    "ingest_pip_version": "2.0",
    "embed_table": "embeddings_table_v2_0"
}
db_database = os.getenv("DB_DATABASE")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")

DB_CONFIG = {
    "dbname": db_database,
    "user": db_user,
    "password": db_password,
    "host": db_host
}

In [4]:
Ingest_Pipeline = IngPipeline(DB_CONFIG,ing_configs)

In [5]:
project_root = os.path.abspath(".")
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)
kbs = os.listdir(ing_configs['kbs_path'])

h:\projects\ai_based\Agent-Factory\src\pipelines


In [6]:
Ingest_Pipeline.ingest_pipeline()


Processing knowledge base: math
Checking document: Calculus - J. Stewart.pdf
  -> Calculus - J. Stewart.pdf already ingested with the same configuration.

Processing knowledge base: physics
Checking document: Mykola Shumskiy - Contract.pdf
  -> Mykola Shumskiy - Contract.pdf already ingested with the same configuration.


In [8]:
Ingest_Pipeline.delete_all_document_entries('Calculus - J. Stewart.pdf')

Deleted ALL 2441 entries for document 'Calculus - J. Stewart.pdf'


2441

In [9]:
Ingest_Pipeline.drop_and_recreate_table()

Dropped existing table: embeddings_table_v2_0
Created new table: embeddings_table_v2_0 with text_hash structure


True

In [9]:
# Recreate the Ingest_Pipeline object with updated configuration
print(f"Current chunk_size in ing_configs: {ing_configs['chunk_size']}")
print(f"Current chunk_size in Ingest_Pipeline object: {Ingest_Pipeline.chunk_size}")

# Recreate the pipeline object to pick up the new chunk_size
Ingest_Pipeline = IngPipeline(DB_CONFIG, ing_configs)
print(f"New chunk_size in Ingest_Pipeline object: {Ingest_Pipeline.chunk_size}")

Current chunk_size in ing_configs: 400
Current chunk_size in Ingest_Pipeline object: 400
New chunk_size in Ingest_Pipeline object: 400


In [9]:
# Test with a single document to verify chunk_size is working correctly
print("Testing single document ingestion with correct chunk_size...")
try:
    result = Ingest_Pipeline.ingest_document('Mykola Shumskiy - Contract.pdf', 'physics')
    print(f"Result type: {type(result)}")
    print(f"Number of chunks created: {len(result) if result else 'None'}")
    if result and len(result) > 0:
        print(f"First chunk size: {result[0]['chunk_size']}")
        print(f"First chunk token count: {result[0]['token_count']}")
except Exception as e:
    print(f"Error during single document ingestion: {e}")

Testing single document ingestion with correct chunk_size...
DEBUG: ingest_document called with document_title='Mykola Shumskiy - Contract.pdf', kb='physics'
Processing document: Mykola Shumskiy - Contract.pdf
Creating chunks...
loading embeddings model...
Creating chunks...
loading embeddings model...
Creating embeddings...
Creating embeddings...
Unloading embeddings model...
Checking table...
Saving embeddings...
Mykola Shumskiy - Contract.pdf has been ingested.
DEBUG: ingest_document completed successfully
Result type: <class 'list'>
Number of chunks created: 15
First chunk size: 400
First chunk token count: 400
Unloading embeddings model...
Checking table...
Saving embeddings...
Mykola Shumskiy - Contract.pdf has been ingested.
DEBUG: ingest_document completed successfully
Result type: <class 'list'>
Number of chunks created: 15
First chunk size: 400
First chunk token count: 400


In [ ]:
# Test with a single document to see if duplication still occurs
print("Testing single document ingestion...")
try:
    result = Ingest_Pipeline.ingest_document('Mykola Shumskiy - Contract.pdf', 'physics')
    print(f"Result type: {type(result)}")
    print(f"Result length: {len(result) if result else 'None'}")
except Exception as e:
    print(f"Error during single document ingestion: {e}")